# Overlap version: what is different from the blocking Jacobi code?

This notebook focuses only on the parts of [src/05-jacobi-overlap.cu](../src/05-jacobi-overlap.cu) that differ from the blocking version in [src/04-jacobi-blocking.cu](../src/04-jacobi-blocking.cu).

The shared setup is the same:
- one MPI rank owns a row slice of the global grid
- each rank keeps ghost rows at the top and bottom
- the stencil is still the five-point Jacobi update

The main difference is the communication pattern.

Instead of doing a blocking MPI exchange with `MPI_Sendrecv`, this code posts nonblocking MPI operations with `MPI_Irecv` and `MPI_Isend`, starts the interior update kernel immediately, and only waits later when the halo data is needed.

That is the overlap: communication and computation happen at the same time.

## 1. Why the overlap version is different

In the blocking version, the code does this in order:

1. exchange ghost rows
2. compute the stencil
3. swap arrays

That is simple and correct, but it serializes communication and computation.

In the overlap version, the code does this instead:

1. start receives and sends for the halos
2. update the interior rows immediately
3. wait for the MPI requests to finish
4. update the first and last local rows that touch the halo data
5. swap arrays

This hides some MPI latency behind the GPU work on the interior cells.

## 2. Nonblocking halo exchange

The key difference is this block:

```cpp
MPI_Request q[4];
MPI_CHECK(MPI_Irecv(
    u,                                    /* top ghost row receive buffer from rank above */
    nx,                                   /* one row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    up,                                   /* source rank above */
    11,                                   /* tag for top halo receive */
    MPI_COMM_WORLD,                       /* communicator */
    &q[0]                                 /* request handle for this receive */
));
MPI_CHECK(MPI_Irecv(
    u + (rows + 1) * nx,                  /* bottom ghost row receive buffer from rank below */
    nx,                                   /* one row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    down,                                 /* source rank below */
    10,                                   /* tag for bottom halo receive */
    MPI_COMM_WORLD,                       /* communicator */
    &q[1]                                 /* request handle for this receive */
));
MPI_CHECK(MPI_Isend(
    u + nx,                               /* first real row to send upward */
    nx,                                   /* one row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    up,                                   /* destination rank above */
    10,                                   /* tag for upward send */
    MPI_COMM_WORLD,                       /* communicator */
    &q[2]                                 /* request handle for this send */
));
MPI_CHECK(MPI_Isend(
    u + rows * nx,                        /* last real row to send downward */
    nx,                                   /* one row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    down,                                 /* destination rank below */
    11,                                   /* tag for downward send */
    MPI_COMM_WORLD,                       /* communicator */
    &q[3]                                 /* request handle for this send */
));
```

The important idea is that these are asynchronous requests, not blocking communication. The code does not stop and wait here.

The request array holds the state of each MPI transfer:
- `q[0]` = receive from above
- `q[1]` = receive from below
- `q[2]` = send to above
- `q[3]` = send to below

## 3. Why the tags are matched this way

The communication is paired by direction and label:

- tag `10` is used for the upward send and the bottom receive
- tag `11` is used for the downward send and the top receive

This is important because the code is exchanging data across two neighboring ranks in opposite directions at the same time.

Without matching tags, a rank could receive the wrong row or confuse which send corresponds to which receive.

## 4. Overlap with interior compute

Once the requests are posted, the code launches the interior update:

```cpp
dim3 gi((nx + 31) / 32, (rows - 2 + 7) / 8);
step_rows<<<gi, b>>>(u, v, nx, 2, rows - 1);
```

This kernel updates rows from `2` through `rows - 1`:

- it excludes the ghost rows
- it excludes the first owned row and the last owned row
- it computes the bulk of the stencil work while MPI transfers are still in flight

That is the core overlap.

## 5. Why the code waits before the boundary rows

After the interior update, the code does:

```cpp
MPI_CHECK(MPI_Waitall(4, q, MPI_STATUSES_IGNORE));
```

This waits until all four nonblocking requests complete.

Then it updates the rows that directly touch the halo data:

```cpp
if (rank > 0) {
  step_rows<<<ge, b>>>(u, v, nx, 1, 1);
}
if (rank < size - 1) {
  step_rows<<<ge, b>>>(u, v, nx, rows, rows);
}
```

These are the boundary rows adjacent to the ghost rows. They must wait for the halo exchange because they read from the newly received neighbor data.

That is why the code does not update them before `MPI_Waitall`.

## 6. Summary of the difference

Compared with the blocking version, the overlap version changes only the communication schedule:

- blocking: communicate fully, then compute
- overlap: communicate asynchronously, compute the interior while communication is in flight, then finish the boundary rows after the wait

This improves performance when the MPI latency is large relative to the cost of computing the interior stencil.

The stencil itself, the ghost layout, and the Jacobi swap pattern are still the same.